In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, sum as spark_sum, count, window
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas10_Sava") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

26/11/09 23:44:28 WARN Utils: Your hostname, tara-Aspire-A514-54G resolves to a loopback address: 127.0.1.1; using 192.168.1.104 instead (on interface wlp4s0)
26/11/09 23:44:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 23:44:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/09 23:44:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [11]:
for q in spark.streams.active:
    q.stop()
    print(f"Query {q.name} dihentikan.")

In [2]:
# A. Menyiapkan Folder dan Source
folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)
print("A. Berhasil menyiapkan folder 'tugas10_stream' dan mendefinisikan streaming source (df_stream).")


# B. Agregasi per Cabang
df_stream_calc = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_cabang = df_stream_calc.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_cabang = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()
print("B. Berhasil membuat dan menjalankan streaming query agregasi per cabang ('agregasi_cabang').")

A. Berhasil menyiapkan folder 'tugas10_stream' dan mendefinisikan streaming source (df_stream).
B. Berhasil membuat dan menjalankan streaming query agregasi per cabang ('agregasi_cabang').


In [3]:
# C. Agregasi berbasis window waktu 2 menit
agregasi_waktu = df_stream_calc.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan")
)

query_waktu = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()
print("C. Berhasil membuat dan menjalankan streaming query berbasis window waktu 2 menit ('agregasi_waktu').")

C. Berhasil membuat dan menjalankan streaming query berbasis window waktu 2 menit ('agregasi_waktu').


In [7]:
# D. Batch 1
pd.DataFrame({
    "pesanan_id": ["P1", "P2"],
    "nama_cabang": ["Cabang Magelang", "Cabang Jogja"],
    "nama_menu": ["Kopi Susu", "Kopi Hitam"],
    "qty": [2, 1],
    "harga_satuan": [15000, 10000],
    "waktu_pesan": ["2026-11-09 23:00:00", "2026-11-09 23:01:00"]
}).to_csv(f"{folder_tugas}/batch1.csv", index=False)

time.sleep(6)
print("--- Hasil Setelah Batch 1 ---")
spark.sql("SELECT * FROM agregasi_cabang").show()

# Batch 2
pd.DataFrame({
    "pesanan_id": ["P3", "P4"],
    "nama_cabang": ["Cabang Magelang", "Cabang Solo"],
    "nama_menu": ["Latte", "Espresso"],
    "qty": [1, 2],
    "harga_satuan": [20000, 12000],
    "waktu_pesan": ["2026-11-09 23:02:00", "2026-11-09 23:03:00"]
}).to_csv(f"{folder_tugas}/batch2.csv", index=False)

time.sleep(6)
print("--- Hasil Setelah Batch 2 ---")
spark.sql("SELECT * FROM agregasi_cabang").show()

# Batch 3
pd.DataFrame({
    "pesanan_id": ["P5", "P6"],
    "nama_cabang": ["Cabang Jogja", "Cabang Solo"],
    "nama_menu": ["Cappuccino", "Mocha"],
    "qty": [2, 1],
    "harga_satuan": [18000, 22000],
    "waktu_pesan": ["2026-11-09 23:04:00", "2026-11-09 23:05:00"]
}).to_csv(f"{folder_tugas}/batch3.csv", index=False)

time.sleep(6)
print("--- Hasil Setelah Batch 3 ---")
spark.sql("SELECT * FROM agregasi_cabang").show()

--- Hasil Setelah Batch 1 ---
+---------------+----------------+--------------+
|    nama_cabang|total_pendapatan|jumlah_pesanan|
+---------------+----------------+--------------+
|   Cabang Jogja|           10000|             1|
|Cabang Magelang|           30000|             1|
+---------------+----------------+--------------+

--- Hasil Setelah Batch 2 ---
+---------------+----------------+--------------+
|    nama_cabang|total_pendapatan|jumlah_pesanan|
+---------------+----------------+--------------+
|Cabang Magelang|           50000|             2|
|   Cabang Jogja|           10000|             1|
|    Cabang Solo|           24000|             1|
+---------------+----------------+--------------+

--- Hasil Setelah Batch 3 ---
+---------------+----------------+--------------+
|    nama_cabang|total_pendapatan|jumlah_pesanan|
+---------------+----------------+--------------+
|Cabang Magelang|           50000|             2|
|   Cabang Jogja|           46000|             2|
|    Cab

In [8]:
# E. Menyiapkan query sink Parquet
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint", ignore_errors=True)

query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

# Mengirim 1 batch tambahan khusus untuk trigger sink parquet
pd.DataFrame({
    "pesanan_id": ["P7"],
    "nama_cabang": ["Cabang Semarang"],
    "nama_menu": ["Kopi Gula Aren"],
    "qty": [3],
    "harga_satuan": [15000],
    "waktu_pesan": ["2026-11-09 23:06:00"]
}).to_csv(f"{folder_tugas}/batch4_parquet.csv", index=False)

time.sleep(6)

print("Bukti file Parquet berhasil dibuat:")
!ls tugas10_output

print("\nMembaca file Parquet kembali sebagai Batch DataFrame:")
df_parquet = spark.read.parquet("tugas10_output")
df_parquet.show()

# Menghentikan semua query di akhir notebook
query_cabang.stop()
query_waktu.stop()
query3.stop()
spark.stop()

Bukti file Parquet berhasil dibuat:
part-00000-58cc99e8-5ca1-49ec-8c65-4b12154fb617-c000.snappy.parquet
part-00001-52d38a58-a3c5-4da2-82d9-5409b8a4a8aa-c000.snappy.parquet
part-00002-3e9f0eff-b1de-48c9-a987-d127dadc416a-c000.snappy.parquet
part-00003-05543bfe-013f-4795-bcb9-7d529fbf88e6-c000.snappy.parquet
_spark_metadata

Membaca file Parquet kembali sebagai Batch DataFrame:
+----------+---------------+--------------+---+------------+-------------------+
|pesanan_id|    nama_cabang|     nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+---------------+--------------+---+------------+-------------------+
|        P7|Cabang Semarang|Kopi Gula Aren|  3|       15000|2026-11-09 23:06:00|
|        P1|Cabang Magelang|     Kopi Susu|  2|       15000|2026-11-09 23:00:00|
|        P2|   Cabang Jogja|    Kopi Hitam|  1|       10000|2026-11-09 23:01:00|
|        P3|Cabang Magelang|         Latte|  1|       20000|2026-11-09 23:02:00|
|        P4|    Cabang Solo|      Espresso|  2|       